# MA12003 Year 1 S2 Programming — Working Notebook

To assess your coursework submission, I will run your **working notebook**.

**Before making any changes to this notebook, rename it to `working_notebook_USERNAME.ipynb`** where `USERNAME` is your university username (e.g. `working_notebook_gh3145.ipynb`).

- Do **not** add or remove any cells in this notebook.
- Write your code in the **[Demonstrate]** cells and your written answers in the **[Justify]** cells.
- All imports must go in the **Imports** section below.

See `coursework_specification.ipynb` for submission instructions, assessment criteria, and task descriptions.

---
#### Enter your full name and Bath user ID (e.g. gh3145) in the cell below:

Name: Joss Nockolds

User ID: jgn26

**GenAI statement:** You must include a short statement (max 250 words) in the cell
below indicating what tools you used and how, or that you have not used genAI.

I used it to generate some tests for various methods. (I came up with a list of required funcitionalities for each class/method/function, before feeding that list into a generative AI, which generated tests for each functionality. I then checked and, where needed, tweaked those tests before using them.)

I used it for bugfixes. (I described the bug to it, fed it my code, and implemented its suggested fix if it was a correct fix.)

I used it to give feedback on my **[Justify]** sections, and I made tweaks to those sections based on its suggestions.

## **Imports**

All imports must appear in the cell below. **Importing packages outside this section will be penalised.**

In [1]:
from abc import ABC, abstractmethod # used in Cipher
import sympy as sp # used in AffineCipher to validate inputs (gcd); used in AffineHillCipher to handle matrices and vectors
from typing import Callable # used for parameter type hints in methods

---
# Part A: Foundational Cipher Implementation &nbsp;&nbsp; [8 marks]

## Task A1: Caesar Cipher $C_c$ &nbsp;&nbsp; [2 marks]

In [2]:
# [Demonstrate]

default_alphabet = "abcdefghijklmnopqrstuvwxyz"

# definitions:

def is_valid_alphabet(alphabet: str) -> bool:
    if not isinstance(alphabet, str):
        return False
    letters = ""
    for letter in alphabet:
        if letter in letters: # letter appears more than once in alphabet
            return False # alphabet should be a subset of "ambient" alphabet
        if letter not in default_alphabet:
            return False # alphabet should be a subset of "ambient" alphabet; handles uppercase issue as well
        letters += letter
    return True

def is_valid_permutation(permutation: str, alphabet: str) -> bool:
    if not isinstance(permutation, str):
        return False
    if not isinstance(alphabet, str):
        return False
    if len(permutation) != len(alphabet): # same length
        return False
    if set(permutation) != set(alphabet): # same elements (alphabet has no repeats, so this and the previous condition cover the case where permutation has duplicates as well)
        return False
    return True

class Cipher(ABC):
    """Abstract base class for all cipher classes"""
    
    @abstractmethod # enforces that all Cipher subclasses must be initialisable
    def __init__(self, alphabet: str = default_alphabet):
        if not is_valid_alphabet(alphabet):
            raise ValueError("alphabet invalid (must be a string of lowercase letters and a subset of the 'ambient' alphabet)")
        self.alphabet = alphabet
        self.alphabet_char_indices = {char: i for (i, char) in enumerate(alphabet)} # dictionary of characters with their indexes

    @abstractmethod
    def __str__(self) -> str:
        ...

    def __repr__(self) -> str:
        return str(self)

    @abstractmethod
    def encode(self, msg: str) -> str:
        ...

    @abstractmethod
    def decode(self, encoded_msg: str) -> str:
        ...

    def print_encryption_decryption(self, msg: str) -> None:
        print(f"Uncoded message: {msg}")
        encoded_msg = self.encode(msg)
        print(f"Encoded message: {encoded_msg}")
        decoded_msg = self.decode(encoded_msg)
        print(f"Decoded message: {decoded_msg}")


class SubstitutionCipher(Cipher):
    def __init__(self, permutation: str | None = None, alphabet: str = default_alphabet):
        super().__init__(alphabet)
        if permutation is None: # identity permutation
            self.alphabet = "" # for consistency
            self.permutation = "" # (as above)
            self.permutation_char_indices = self.alphabet_char_indices
            return # early exit, because the cipher is fully defined already
            
        if not is_valid_permutation(permutation, alphabet):
            raise ValueError("permutation invalid (must be a string of lowercase letters which permutes alphabet)")
        self.permutation = permutation
        self.permutation_char_indices = {char: i for (i, char) in enumerate(permutation)} # dictionary of characters, with their indexes in `permutation`
        
    def __str__(self) -> str:
        return f"Substitution cipher ({self.alphabet} ↦ {self.permutation})"

    def encode(self, msg: str) -> str:
        if not isinstance(msg, str):
            raise TypeError("msg must be an str")
        return "".join(self._map_char(char, encode = True) for char in msg)

    def decode(self, encoded_msg: str) -> str:
        if not isinstance(encoded_msg, str):
            raise TypeError("encoded_msg must be an str")
        return "".join(self._map_char(char, encode = False) for char in encoded_msg)

    # Helpher methods:
    
    def _map_char(self, char: str, encode: bool = True) -> str:
        if not isinstance(char, str):
            raise TypeError("char must be a str")
        if len(char) > 1:
            raise ValueError("char must be of length 1 (i.e. a single character)")
    
        lower_char = char.lower()
        if encode == True: # encode maps from alphabet to permutation
            indices = self.alphabet_char_indices
            destination = self.permutation
        else: # decode maps from permutation to alphabet instead
            indices = self.permutation_char_indices
            destination = self.alphabet

        index = indices.get(lower_char)
        if index is None: # if the character isn't in the cipher alphabet/permutation
            return char # keep the character the same
        
        if char == lower_char: # if char is lowercase
            return destination[index]
        else:
            return destination[index].upper()

        
class CaesarCipher(SubstitutionCipher):
    def __init__(self, shift: int, alphabet: str = default_alphabet):
        if not isinstance(shift, int):
            raise TypeError("shift must be an int")
        self.alphabet = alphabet # needed for the permutation calculation
        self.alphabet_char_indices = {char: i for (i, char) in enumerate(alphabet)} # needed for the permutation calculation
        self.shift = shift % len(alphabet) # keeps values in interval [0, len(alphabet)), for consistency when comparing ciphers
        super().__init__(self._get_caesar_permutation(), alphabet)

    def __str__(self) -> str:
        return f"Caesar cipher (shift: {self.shift}, alphabet: {self.alphabet})"

    # Helper methods:
    
    def _encode_alphabet_char_caesar(self, char: str) -> str:
        # doesn't need to check for char's type, because this function is only used on alphabets
        # doesn't need to support uppercase letters, because it'll only be used on alphabets
        old_index = self.alphabet_char_indices.get(char)
        # doesn't need to support letters not in the alphabet, since it'll only be used on alphabets 
        new_index = (old_index + self.shift) % len(self.alphabet) # mod prevents index underflows and overflows
        return self.alphabet[new_index]

    def _get_caesar_permutation(self) -> str:
        return "".join(self._encode_alphabet_char_caesar(char) for char in self.alphabet) # encode the alphabet to get the permutation

        

# encoding and decoding the six secret messages:

bond = 'Agent 007: "Shaken, not stirred!"'
holmes = '[At 221B Baker Street] Holmes (to Watson): "Elementary, my dear Watson!"'
joker = "Why so serious?! Let's put a smile on that face."
turing = 'In 1950, Turing asked: "Can machines think?"'
hamlet = "[Act 3, Scene 1] To be, or not to be?"
enigma = "[Bletchley, 1941] The Enigma has been cracked!"
messages = [bond, holmes, joker, turing, hamlet, enigma]

border = "========================================"

C_3 = CaesarCipher(3)
C_neg5 = CaesarCipher(-5)

print(border)
print(C_3)
print(border)

for message in messages:
    C_3.print_encryption_decryption(message)
    print(border)

print() # spacing

print(border)
print(C_neg5)
print(border)

for message in messages:
    C_neg5.print_encryption_decryption(message)
    print(border)


Caesar cipher (shift: 3, alphabet: abcdefghijklmnopqrstuvwxyz)
Uncoded message: Agent 007: "Shaken, not stirred!"
Encoded message: Djhqw 007: "Vkdnhq, qrw vwluuhg!"
Decoded message: Agent 007: "Shaken, not stirred!"
Uncoded message: [At 221B Baker Street] Holmes (to Watson): "Elementary, my dear Watson!"
Encoded message: [Dw 221E Ednhu Vwuhhw] Krophv (wr Zdwvrq): "Hohphqwdub, pb ghdu Zdwvrq!"
Decoded message: [At 221B Baker Street] Holmes (to Watson): "Elementary, my dear Watson!"
Uncoded message: Why so serious?! Let's put a smile on that face.
Encoded message: Zkb vr vhulrxv?! Ohw'v sxw d vploh rq wkdw idfh.
Decoded message: Why so serious?! Let's put a smile on that face.
Uncoded message: In 1950, Turing asked: "Can machines think?"
Encoded message: Lq 1950, Wxulqj dvnhg: "Fdq pdfklqhv wklqn?"
Decoded message: In 1950, Turing asked: "Can machines think?"
Uncoded message: [Act 3, Scene 1] To be, or not to be?
Encoded message: [Dfw 3, Vfhqh 1] Wr eh, ru qrw wr eh?
Decoded message: [Ac

## **[Justify]** 
I've gone for an objected-oriented design, both because of `SubstitutionCipher`'s extendability to the other two ciphers in part A, and because it makes it easier to deal with compositions and operations later on (since I can instantiate new ciphers for compositions and just overload operators).

`Cipher` is an abstract class because it isn't defined enough to encode/decode anything itself, so someone shouldn't be able to instantiate it. It holds the basic functionality (encoding, decoding, and representing) that all ciphers should have. `print_encryption_decryption()` was just defined to make outputting results easier. I defined `__repr__()` to return `str(self)`, which is generally best practise.

`SubstitutionCipher` inherits from `Cipher` and has the functionality to substitute the characters in a message for other characters, according to its permutation and alphabet. This means that I can define any possible character-by-character substitition cipher, as long as the correct permutation is inputted, allowing for maximum flexibility and reusability. For simplicity, no inputs when instantiating means it's an identity "cipher".

In its constructor, I've defined the `alphabet_char_indices` dictionary (using dictionary comprehention and `enumerate()`) to access the alphabet index of characters in $O(1)$ time, instead of `str.index()`'s $O($`len(alphabet)`$)$ time, so it's more efficient when encoding (the same is true for decoding, using `permutation_char_indices`).

For each character in a given message, `_map_char(char, encode=True)` finds its index in `alphabet` (or the index of its lowercase version, using `char.lower()`) (if it can't find the index, the character is left unchanged because it's not in the cipher alphabet), and gets the corresponding character in `permutation`, using `permutation[i]` (or `permutation[i].upper()`, if it was uppercase), thereby mapping the character. (Note that, by Python's conventions, an underscore before a class' attribute or method name means that it's private, and ideally shouldn't be accessed from outside of the class.) And it works in reverse for decoding, with `_map_char(char, encode=False)` changing behaviour based on a boolean, instead of a whole new method, to reduce repeated code.

`encode()` uses a generator comprehension on `msg`, with `_map_char(char, encode=True)`, to efficiently encode all of `msg`'s characters, before converting it to a string with `"".join(...)`. `decode()` does the same, but with `_map_char(char, encode=False)`.

`CaesarCipher` inherits from `SubstitutionCipher` because it is a substitution cipher, so all of its substitution logic can be focused in one parent class, instead of redefined. I defined its `permutation` by Caesar-shifting `alphabet`. This way, I didn't need to speicfically make custom encode/decode methods for it, which would be unnecessary bloat and repeated code. It also means that I could completely avoid having a shift calculation happening for each character. Note that I normalise `shift` to `shift % len(alphabet)`, for consistency when comparing ciphers (since $C_c = C_{c mod n}$).

Because `encode()` only works after the permutation is defined, I had to implement `_get_caesar_permutation()`, which is similar to `encode()`, but specifically for a Caesar cipher. Because it only gets used on alphabets, I could drop the input validation, uppercase handling, and non-alphabet-character handling, making it more efficient (and it is "private", because of its name's leading underscore, so it shouldn't be used in other cases).

To decode a Caesar cipher of any shift value, my code simply swaps the roles of the permutation and alphabet, so that each character is mapped back to its origin. For example, with $C_3$, during instantiation, `permutation` is defined as `defg...yzabc` (by shifting its characters using `_get_caesar_permutation()`, which uses `_encode_alphabet_char_caesar()` on each character), and the character indices dictionaries are defined accordingly. When decoding, we use `permutation_char_indices.get(char)` to get the index `i` of each character `char`, and then we use `alphabet[i]` to get its original character (or `alphabet[i].upper()`, if it was uppercase). So, if the current `char` was `"f"`, the index `i` would be `2`, and its original character would therefore be `"c"`. This process is the same as encoding it with shift `-c`.

To make sure that nobody can use the cipher classes/methods incorrectly, throughout my code I've made sure to validate any parameters that could be passed in incorrectly by raising errors (with relevant messages) if a parameter is invalid (where necessary). Particularly, I've made sure that alphabets are accurate, according to the specification's definition, using `is_valid_alphabet()`, and that any permutation makes sense with its corresponding alphabet, using `is_valid_permutation()`. The latter uses `set()` and `len()` on both `alphabet` and `permutation` to validate efficiently (same length and same elements means they permute each other). I've also used type hints for parameters and return types for readability and to clearly define the expected input types.

## Task A2: Affine Cipher $A_{a,b}$ &nbsp;&nbsp; [2 marks]

In [3]:
# [Demonstrate]

# definitions:

class AffineCipher(SubstitutionCipher):
    """[Placeholder]"""
    
    def __init__(self, a: int, b: int, alphabet: str = default_alphabet):
        if not isinstance(a, int):
            raise TypeError("a must be an int")
        if not isinstance(b, int):
            raise TypeError("b must be an int")
        l = len(alphabet)
        if sp.gcd(a, l) != 1:
            raise ValueError("a and len(alphabet) must be coprime (otherwise the cipher isn't decodable)")
        self.alphabet = alphabet # needed for the permutation calculation
        self.alphabet_char_indices = {char: i for (i, char) in enumerate(alphabet)} # needed for the permutation calculation
        self.a = a % l # keeps values in interval [0, l), for consistency when comparing ciphers
        self.b = b % l # (as above)
        super().__init__(self._get_affine_permutation(), alphabet)


    def __str__(self) -> str:
        return f"Affine cipher (i ↦ {self.a}i + {self.b}, alphabet: {self.alphabet})"
    
    # Helper methods:

    def _encode_alphabet_char_affine(self, char: str) -> str:
        # doesn't need to check for char's type, because this function is only used on alphabets
        # doesn't need to support uppercase letters, because it'll only be used on alphabets
        old_index = self.alphabet_char_indices.get(char)
        # doesn't need to support letters not in the alphabet, since it'll only be used on alphabets 
        new_index = (self.a * old_index + self.b) % len(self.alphabet) # mod prevents index underflows and overflows
        return self.alphabet[new_index]

    def _get_affine_permutation(self) -> str:
        return "".join(self._encode_alphabet_char_affine(char) for char in self.alphabet) # encode the alphabet to get the permutation


# encoding and decoding the six secret messages:

A_7_3 = AffineCipher(7, 3)
A_5_8 = AffineCipher(5, 8)

print(border)
print(A_7_3)
print(border)

for message in messages:
    A_7_3.print_encryption_decryption(message)
    print(border)

print() # spacing

print(border)
print(A_5_8)
print(border)

for message in messages:
    A_5_8.print_encryption_decryption(message)
    print(border)


Affine cipher (i ↦ 7i + 3, alphabet: abcdefghijklmnopqrstuvwxyz)
Uncoded message: Agent 007: "Shaken, not stirred!"
Encoded message: Dtfqg 007: "Zadvfq, qxg zghssfy!"
Decoded message: Agent 007: "Shaken, not stirred!"
Uncoded message: [At 221B Baker Street] Holmes (to Watson): "Elementary, my dear Watson!"
Encoded message: [Dg 221K Kdvfs Zgsffg] Axcjfz (gx Bdgzxq): "Fcfjfqgdsp, jp yfds Bdgzxq!"
Decoded message: [At 221B Baker Street] Holmes (to Watson): "Elementary, my dear Watson!"
Uncoded message: Why so serious?! Let's put a smile on that face.
Encoded message: Bap zx zfshxnz?! Cfg'z eng d zjhcf xq gadg mdrf.
Decoded message: Why so serious?! Let's put a smile on that face.
Uncoded message: In 1950, Turing asked: "Can machines think?"
Encoded message: Hq 1950, Gnshqt dzvfy: "Rdq jdrahqfz gahqv?"
Decoded message: In 1950, Turing asked: "Can machines think?"
Uncoded message: [Act 3, Scene 1] To be, or not to be?
Encoded message: [Drg 3, Zrfqf 1] Gx kf, xs qxg gx kf?
Decoded message: [

## **[Justify]**
`AffineCipher`'s implementation is almost identical to `CaesarCipher`'s, but with attributes `a` and `b` instead of `shift`, and mapping indices with `(self.a * old_index + self.b) % len(self.alphabet)` instead of `(old_index + self.shift) % len(self.alphabet)`. I define the permutation during instantiation and rely on `SubstitutionCipher` for encoding and decoding as discussed in Task A1, but using `_get_affine_permutation()` and `_encode_alphabet_char_affine()` to create the permutation instead.

Importantly though, this cipher requires `gcd(a, len(alphabet)) == 1`, because, if not, then `permutation` wouldn't actually be a permutation of `alphabet` (multiple characters would map to the same character), so it would be impossible to decode. This is because essentially what `SubstitutionCipher`'s swap of `alphabet` and `permutation` is doing here is mapping $i$ to $a^{-1}(i-b)$ (because that inverts the affine encode map $i \mapsto ai+b$). But the inverse of `a`, modulo `len(alphabet)` only exists if `a` and `len(alphabet)` are coprime, hence the restriction on `a`.

For example, if `a = 2`, `b = 1`, and `alphabet = "abcd"`, we get `permutation = "bdbd"`, which is clearly invalid (and it's caught by `is_valid_permutation()`). But, if we instead have `a = 3`, we get `permutation = "badc"`, which _is_ valid. 

Meanwhile, `b`'s value is unrestricted (besides it being an integer), because it's simply a shift.

## Task A3: Substitution Cipher $S_p$ &nbsp;&nbsp; [2 marks]

In [4]:
# [Demonstrate]

# adding the capability for SubstitutionCipher to take in functions as permutations:

permutation_functions = set()
# defining a decorator so that SubstitutionCipher can handle permutation input validation
def permutation(function: Callable[..., str]):
    def wrapper(alphabet: str, *args) -> str: # handle errors each time the function is called
        if not isinstance(alphabet, str):
            raise TypeError("alphabet must be a str")
        return function(alphabet, *args)
    permutation_functions.add(wrapper) # add function only when its defined, not every time it's called
    return wrapper

reverse = permutation(lambda alphabet: alphabet[::-1])

@permutation
def split(alphabet: str) -> str:
    evens = alphabet[::2]
    if len(alphabet) % 2 == 0: # if even length, then the last index will be an odd index
        odds_reversed = alphabet[-1:0:-2] # so we start on the last index
    else: # if not, then the last index will be an even index
        odds_reversed = alphabet[-2:0:-2] # so we start on the second last index
    return evens + odds_reversed
    
class SubstitutionCipher(Cipher):
    """[Placeholder]"""
    def __init__(self, permutation: str | Callable[..., str] | None = None, alphabet: str = default_alphabet):
        super().__init__(alphabet)
        
        # identity permutation:
        if permutation is None:
            self.alphabet = "" # for consistency
            self.permutation = "" # (as above)
            self.permutation_char_indices = self.alphabet_char_indices
            return # early exit, because the cipher is fully defined already
            
        # function-defined permutation:
        if callable(permutation) and (permutation in permutation_functions): # only accepts valid permutations
            self.permutation = permutation(self.alphabet) # encode the alphabet to get the string permutation
            self.permutation_char_indices = {char: i for (i, char) in enumerate(self.permutation)} # dictionary of characters with their indexes in `permutation`
            return # early exit, because the cipher is fully defined already

        # string-defined permutation:
        if not is_valid_permutation(permutation, alphabet):
            raise ValueError("permutation invalid (must be a string of lowercase letters which permutes alphabet, or a @permutation function)")
        self.permutation = permutation
        self.permutation_char_indices = {char: i for (i, char) in enumerate(permutation)} # dictionary of characters with their indexes in `permutation`
        
    def __str__(self) -> str:
        return f"Substitution cipher ({self.alphabet} ↦ {self.permutation})"

    def encode(self, msg: str) -> str:
        if not isinstance(msg, str):
            raise TypeError("msg must be an str")
        return "".join(self._map_char(char) for char in msg)

    def decode(self, encoded_msg: str) -> str:
        if not isinstance(encoded_msg, str):
            raise TypeError("encoded_msg must be an str")
        return "".join(self._map_char(char, encode = False) for char in encoded_msg)

    # Helpher methods:
    
    def _map_char(self, char: str, encode: bool = True) -> str:
        if not isinstance(char, str):
            raise TypeError("char must be a str")
        if len(char) > 1:
            raise ValueError("char must be of length 1 (i.e. a single character)")
    
        lower_char = char.lower()
        if encode == True: # encode maps from alphabet to permutation
            indices = self.alphabet_char_indices
            destination = self.permutation
        else: # decode maps from permutation to alphabet instead
            indices = self.permutation_char_indices
            destination = self.alphabet

        index = indices.get(lower_char)
        if index is None: # if the character isn't in the cipher alphabet/permutation
            return char # keep the character the same
        
        if char == lower_char: # if char is lowercase
            return destination[index]
        else:
            return destination[index].upper()

# encoding and decoding the six secret messages:
S_reverse = SubstitutionCipher(reverse)
S_split = SubstitutionCipher(split)

print(border)
print(S_reverse)
print(border)

for message in messages:
    S_reverse.print_encryption_decryption(message)
    print(border)

print() # spacing

print(border)
print(S_split)
print(border)

for message in messages:
    S_split.print_encryption_decryption(message)
    print(border)
    

Substitution cipher (abcdefghijklmnopqrstuvwxyz ↦ zyxwvutsrqponmlkjihgfedcba)
Uncoded message: Agent 007: "Shaken, not stirred!"
Encoded message: Ztvmg 007: "Hszpvm, mlg hgriivw!"
Decoded message: Agent 007: "Shaken, not stirred!"
Uncoded message: [At 221B Baker Street] Holmes (to Watson): "Elementary, my dear Watson!"
Encoded message: [Zg 221Y Yzpvi Hgivvg] Slonvh (gl Dzghlm): "Vovnvmgzib, nb wvzi Dzghlm!"
Decoded message: [At 221B Baker Street] Holmes (to Watson): "Elementary, my dear Watson!"
Uncoded message: Why so serious?! Let's put a smile on that face.
Encoded message: Dsb hl hvirlfh?! Ovg'h kfg z hnrov lm gszg uzxv.
Decoded message: Why so serious?! Let's put a smile on that face.
Uncoded message: In 1950, Turing asked: "Can machines think?"
Encoded message: Rm 1950, Gfirmt zhpvw: "Xzm nzxsrmvh gsrmp?"
Decoded message: In 1950, Turing asked: "Can machines think?"
Uncoded message: [Act 3, Scene 1] To be, or not to be?
Encoded message: [Zxg 3, Hxvmv 1] Gl yv, li mlg gl yv?
Decod

## **[Justify]**
The `SubstitutionCipher` class I defined in Task A1 had no simple way of defining a permutation like $\text{reverse}$ or $\text{split}$. So I defined those as functions (using a `lambda` for `reverse()`, because its implementation was one line). That way, you don't have to, for example, manually input `"zyx...cba"` as `permutation` if you want to use $S_{\text{reverse}}$; you can instead just pass in the `reverse` function to the constructor. This also allows new permutation functions to be defined easily.

I defined `permutation()` as a wrapper, which adds each function to `permutation_functions` (a set, because order doesn't matter and duplicates don't add anything) when defined, and does the error handling on the functions' `alphabet` parameters, all in one place. I can then check if a given permutation function, passed into `SubstitutionCipher`'s constructor, is a valid permutation, and raise an error if not (to prevent arbitrary callables, like `print()`, from being passed in). A valid permutation function should act directly on an alphabet, mapping it onto a valid permutation, which then defines `SubstitutionCipher`'s `permutation` attribute.

This way, there are three ways to instantiate a `SubstitutionCipher` object: pass in a permutation string, pass in a permutation function, and pass in nothing, allowing for greater flexibility and reusability. In the last case, the cipher acts as the identity "cipher", because having a standard identity cipher will be useful in later sections. 

To decode a substitution cipher, I go through the same process as encoding, but swapping the roles of `alphabet` and `permutation`, as explained in Task A1. For example, for $S_{\text{reverse}}$, `permutation` is set to `"zyx...cba"`. So, to decode, say, the string `"svool"`, we first get the character corresponding to `"s"` in `alphabet` (namely `"h"`). We repeat this process for each character using a generator comprehension, and then `"".join` to turn that into a string, giving us the decoded string `"hello"`.

## Task A4: Custom Alphabets &nbsp;&nbsp; [2 marks]

In [5]:
# [Demonstrate]

vowels = "aeiou"
consonants = "bcdfghjklmnpqrstvwxyz"
qwerty = "qwertyuiopasdfghjklzxcvbnm"
ambidextrous = "ambidextrous"

C_3_vowels = CaesarCipher(3, vowels)
A_2_5_consonants = AffineCipher(2, 5, consonants)
S_reverse_qwerty = SubstitutionCipher(reverse, qwerty)
S_split_ambidextrous = SubstitutionCipher(split, ambidextrous)
ciphers = [C_3_vowels, A_2_5_consonants, S_reverse_qwerty, S_split_ambidextrous]

for cipher in ciphers:
    print(border)
    print(cipher)
    print(border)
    for message in messages:
        cipher.print_encryption_decryption(message)
        print(border)
    print() # spacing


Caesar cipher (shift: 3, alphabet: aeiou)
Uncoded message: Agent 007: "Shaken, not stirred!"
Encoded message: Ogunt 007: "Shokun, net starrud!"
Decoded message: Agent 007: "Shaken, not stirred!"
Uncoded message: [At 221B Baker Street] Holmes (to Watson): "Elementary, my dear Watson!"
Encoded message: [Ot 221B Bokur Struut] Helmus (te Wotsen): "Ulumuntory, my duor Wotsen!"
Decoded message: [At 221B Baker Street] Holmes (to Watson): "Elementary, my dear Watson!"
Uncoded message: Why so serious?! Let's put a smile on that face.
Encoded message: Why se suraeis?! Lut's pit o smalu en thot focu.
Decoded message: Why so serious?! Let's put a smile on that face.
Uncoded message: In 1950, Turing asked: "Can machines think?"
Encoded message: An 1950, Tirang oskud: "Con mochanus thank?"
Decoded message: In 1950, Turing asked: "Can machines think?"
Uncoded message: [Act 3, Scene 1] To be, or not to be?
Encoded message: [Oct 3, Scunu 1] Te bu, er net te bu?
Decoded message: [Act 3, Scene 1] To be, 

## **[Justify]**
In my base `Cipher` class, I store the parameter `alphabet` as an attribute. All other cipher classes in A1, A2, and A3 are subclasses (or sub-subclases) of `Cipher` and use `super().__init__()` to carry over this behaviour with no unnecessary code repetition. And in my code, I only reference `self.alphabet`, so any alphabet should work (as long as it's passed into the cipher's constructor). I even implemented input validation to check if any given alphabet is valid, as discussed in Task A1.

The default case uses the $\text{default}$ alphabet because it covers all characters in the Latin alphabet. So any standard English would be fully encoded by default.

For convenience, where an alphabet is a parameter of `__init__()`, I've used `alphabet = default_alphabet` so that users who want to fully encode a message don't have to type out `default_alphabet` each time, mirroring how the specification defines no provided alphabet to mean that the alphabet is $\text{default}$.

---
# Part B: Cipher Manipulation and Composition &nbsp;&nbsp; [8 marks]

## Task B1: Operations on Caesar Ciphers &nbsp;&nbsp; [2 marks]

In [6]:
# [Demonstrate]

# adding composition support for SubstitutionCipher:
class SubstitutionCipher(Cipher):
    """[Placeholder]"""
    def __init__(self, permutation: str | Callable[..., str] | None = None, alphabet: str = default_alphabet):
        super().__init__(alphabet)
        
        # identity permutation:
        if permutation is None:
            self.alphabet = "" # for consistency
            self.permutation = "" # (as above)
            self.permutation_char_indices = self.alphabet_char_indices
            return # early exit, because the cipher is fully defined already
            
        # function-defined permutation:
        if callable(permutation) and (permutation in permutation_functions): # only accepts valid permutations
            self.permutation = permutation(self.alphabet) # encode the alphabet to get the string permutation
            self.permutation_char_indices = {char: i for (i, char) in enumerate(self.permutation)} # dictionary of characters with their indexes in `permutation`
            return # early exit, because the cipher is fully defined already

        # string-defined permutation:
        if not is_valid_permutation(permutation, alphabet):
            raise ValueError("permutation invalid (must be a string of lowercase letters which permutes alphabet, or a @permutation function)")
        self.permutation = permutation
        self.permutation_char_indices = {char: i for (i, char) in enumerate(permutation)} # dictionary of characters with their indexes in `permutation`
        
    def __str__(self) -> str:
        return f"Substitution cipher ({self.alphabet} ↦ {self.permutation})"

    def encode(self, msg: str) -> str:
        if not isinstance(msg, str):
            raise TypeError("msg must be an str")
        return "".join(self._map_char(char) for char in msg)

    def decode(self, encoded_msg: str) -> str:
        if not isinstance(encoded_msg, str):
            raise TypeError("encoded_msg must be an str")
        return "".join(self._map_char(char, encode = False) for char in encoded_msg)

    # Helpher methods:
    
    def _map_char(self, char: str, encode: bool = True) -> str:
        if not isinstance(char, str):
            raise TypeError("char must be a str")
        if len(char) > 1:
            raise ValueError("char must be of length 1 (i.e. a single character)")
    
        lower_char = char.lower()
        if encode == True: # encode maps from alphabet to permutation
            indices = self.alphabet_char_indices
            destination = self.permutation
        else: # decode maps from permutation to alphabet instead
            indices = self.permutation_char_indices
            destination = self.alphabet

        index = indices.get(lower_char)
        if index is None: # if the character isn't in the cipher alphabet/permutation
            return char # keep the character the same
        
        if char == lower_char: # if char is lowercase
            return destination[index]
        else:
            return destination[index].upper()

    # Operator overloads:
    
    def __matmul__(self, other) -> "SubstitutionCipher":
        if not isinstance(other, SubstitutionCipher):
            raise TypeError(f"Can only compose (@) SubstitutionCipher with SubstitutionCipher")
        if other.alphabet == other.permutation: # if identity; it means we don't have to do the re-encoding below, which is O(len(new_alphabet))
            return self
        if self.alphabet == self.permutation: # (as above)
            return other
        # combine both alphabets, remove duplicates, and (for consistency and readability) sort alphabetically:
        new_alphabet = ''.join(sorted(set(self.alphabet + other.alphabet))) 
        new_permutation = self.encode(other.encode(new_alphabet))
        return SubstitutionCipher(new_permutation, new_alphabet)

    def __pow__(self, other) -> "SubstitutionCipher":
        if not isinstance(other, int):
            raise TypeError(f"Can only raise (**) SubstitutionCipher to the power of an int")
        if other < 0:
            raise ValueError(f"Can only raise (**) SubstitutionCipher to a non-negative power")
        if other == 0:
            return SubstitutionCipher() # identity "cipher"
        new_permutation = self.permutation
        for _ in range(other - 1): # for a total of `other` instances of `self`'s cipher
            new_permutation = self.encode(new_permutation)
        return SubstitutionCipher(new_permutation, self.alphabet)


# adding operator overloads for CaesarCipher:
class CaesarCipher(SubstitutionCipher):
    """[Placeholder]"""
    
    def __init__(self, shift: int, alphabet: str = default_alphabet):
        if not isinstance(shift, int):
            raise TypeError("shift must be an int")
        self.alphabet = alphabet # needed for the permutation calculation
        self.alphabet_char_indices = {char: i for (i, char) in enumerate(alphabet)} # needed for the permutation calculation
        self.shift = shift % len(alphabet) # keeps values in interval [0, len(alphabet)), for consistency when comparing ciphers
        super().__init__(self._get_caesar_permutation(), alphabet)

    def __str__(self) -> str:
        return f"Caesar cipher (shift: {self.shift}, alphabet: {self.alphabet})"

    # Helper methods:
    
    def _encode_alphabet_char_caesar(self, char: str):
        # doesn't need to check for char's type, because this function is only used on alphabets
        # doesn't need to support uppercase letters, because it'll only be used on alphabets
        old_index = self.alphabet_char_indices.get(char)
        # doesn't need to support letters not in the alphabet, since it'll only be used on alphabets 
        new_index = (old_index + self.shift) % len(self.alphabet) # mod prevents index underflows and overflows
        return self.alphabet[new_index]

    def _get_caesar_permutation(self):
        return "".join(self._encode_alphabet_char_caesar(char) for char in self.alphabet) # encode the alphabet to get the permutation

    # Operator overloads:
    
    def __add__(self, other) -> "CaesarCipher":
        if not isinstance(other, CaesarCipher):
            raise TypeError(f"Can only add (+) CaesarCipher to CaesarCipher")
        if self.alphabet != other.alphabet:
            raise ValueError("Operands must have the same alphabet")
        return CaesarCipher(self.shift + other.shift, self.alphabet)
        
    def __neg__(self) -> "CaesarCipher":
        return CaesarCipher(-self.shift, self.alphabet)
        
    def __sub__(self, other) -> "CaesarCipher":
        if not isinstance(other, CaesarCipher):
            raise TypeError(f"Can only subtract (-) CaesarCipher from CaesarCipher")
        # other errors are handled by __add__ already
        return self + (-other)
        
    def __mul__(self, other) -> "CaesarCipher":
        if not isinstance(other, int):
            raise TypeError(f"Can only multiply (*) CaesarCipher by int")
        return CaesarCipher(self.shift * other, self.alphabet)

    def __rmul__(self, other) -> "CaesarCipher":
        return self * other

    def __matmul__(self, other) -> SubstitutionCipher:
        # error handling is handled by SubstitutionCipher.__matmul__
        if isinstance(other, CaesarCipher) and (self.alphabet == other.alphabet):
            return self + other # C_a ∘ C_b = C_{a+b}, since both performs shifts of a+b in total
        return SubstitutionCipher.__matmul__(self, other) # use SubstitutionCipher's @ instead
        
    def __pow__(self, other) -> "SubstitutionCipher":
        if not isinstance(other, int):
            raise TypeError(f"Can only raise (**) CaesarCipher to the power of an int")
        if other < 0:
            raise ValueError(f"Can only raise (**) CaesarCipher to a non-negative power")
        if other == 0:
            return SubstitutionCipher() # identity "cipher"
        return other * self # a * C_b = (C_b)^a, since both performs shifts of a*b in total

    # Helper methods:

    def _caesar_permutation_alphabet_char(char: str, shift: int, alphabet: str, alphabet_char_indices: dict[str, int]):
        # doesn't need to check for char's type, because this function is only used on alphabets
        # doesn't need to support uppercase letters, because it'll only be used on alphabets
        old_index = alphabet_char_indices.get(char)
        # doesn't need to support letters not in the alphabet, since it'll only be used on alphabets 
        new_index = (old_index + shift) % len(alphabet) # mod prevents index underflows and overflows
        return alphabet[new_index]

    def _caesar_permutation_alphabet(shift: int, alphabet: str, alphabet_char_indices: dict[str, int]):
        return "".join(CaesarCipher._caesar_permutation_alphabet_char(char, shift, alphabet, alphabet_char_indices) for char in alphabet)



# encoding and decoding the six secret messages:

SubstitutionCipher(reverse)
C_3_vowels = CaesarCipher(3, vowels)
C_neg5_vowels = CaesarCipher(-5, vowels)
C_3_ambidextrous = CaesarCipher(3, ambidextrous)
C_2 = CaesarCipher(2)
C_2_vowels = CaesarCipher(2, vowels)
C_3_qwerty = CaesarCipher(3, qwerty)
C_4_qwerty = CaesarCipher(4, qwerty)
C_5_consonants = CaesarCipher(5, consonants)

cipher1 = C_3_vowels @ C_neg5_vowels
cipher2 = C_3_ambidextrous ** 4
cipher3 = 5 * C_2
cipher4 = -C_2_vowels
cipher5 = C_3_qwerty + C_4_qwerty
cipher6 = C_3_vowels @ C_5_consonants
ciphers = [cipher1, cipher2, cipher3, cipher4, cipher5, cipher6]

for cipher in ciphers:
    print(border)
    print(cipher)
    print(border)
    cipher.print_encryption_decryption(default_alphabet)
    for message in messages:
        cipher.print_encryption_decryption(message)
        print(border)
    print() # spacing


Caesar cipher (shift: 3, alphabet: aeiou)
Uncoded message: abcdefghijklmnopqrstuvwxyz
Encoded message: obcdufghajklmnepqrstivwxyz
Decoded message: abcdefghijklmnopqrstuvwxyz
Uncoded message: Agent 007: "Shaken, not stirred!"
Encoded message: Ogunt 007: "Shokun, net starrud!"
Decoded message: Agent 007: "Shaken, not stirred!"
Uncoded message: [At 221B Baker Street] Holmes (to Watson): "Elementary, my dear Watson!"
Encoded message: [Ot 221B Bokur Struut] Helmus (te Wotsen): "Ulumuntory, my duor Wotsen!"
Decoded message: [At 221B Baker Street] Holmes (to Watson): "Elementary, my dear Watson!"
Uncoded message: Why so serious?! Let's put a smile on that face.
Encoded message: Why se suraeis?! Lut's pit o smalu en thot focu.
Decoded message: Why so serious?! Let's put a smile on that face.
Uncoded message: In 1950, Turing asked: "Can machines think?"
Encoded message: An 1950, Tirang oskud: "Con mochanus thank?"
Decoded message: In 1950, Turing asked: "Can machines think?"
Uncoded message: [A

## **[Justify]**
Because I made my ciphers all objects in Task A1, I just overloaded `CaesarCipher`'s arithmetic operators to modify and combine ciphers. For example, for addition, I overloaded `__add__`, which returns `CaesarCipher(self.shift + other.shift, self.alphabet)`, following the specification's definition. I handled scalar multiplication and negation similarly, overloading `__mul__` (and `__rmul__`, to allow for `int * CaesarCipher` on top of `CaesarCipher * int`) and `__neg__` respectively, with suitable return values (and `__sub__` simply returns `self + (-other)`). To ensure the binary operations make sense, I validated inputs to ensure that `other` cipher has to be a `CaesarCipher` and has to have the same alphabet as `self`, raising suitable errors if that isn't the case (with `__sub__` relying on `__add__`'s validation).

For compositions of Caesar ciphers, I overloaded `__matmul__` (or `@`) because it's used for other non-commutative operations, like matrix multiplication, so it feels appropriate for compositions (and using `__mul__` would cause ambiguity between composition and scalar multiplication). (Note that while compositions of Caesar ciphers on the same alphabet _are_ commutative, the same is not true for ciphers in general.) For Caesar ciphers on the same alphabet, it just adds the shifts together, because shifting by $c_1$ and then $c_2$ is the same as shifting by $c_1 + c_2$. However, if the alphabets or cipher types of the ciphers are different, it defaults to `SubstitutionCipher`'s `__matmul__`, which handles all `SubstitutionCipher` compositions, described below.

For powers of Caesar ciphers, I overloaded `__pow__`. After validating that `other` is a valid exponent (raising errors if it isn't a non-negative integer), I just multiply the whole cipher by `other`. This is because both operations are identical ($C_c^n = nC_c$), as they both perform a total shift of `other * c`.

In `SubstitutionCipher`, I overloaded `__matmul__` to compose with any other `SubstitutionCipher` (raising an error if the other cipher isn't a `SubstitutionCipher`). If either cipher is the identity cipher, (determined by `alphabet == permutation`), it returns the other cipher, because no composition changes need to be made. Otherwise, it combines the `alphabet`s of both ciphers (using `set()` to remove duplicates and `"".join()` to convert it back into a string), gets the new permutation by using `self.encode(other.encode())` to encode the new alphabet twice, and returns the new cipher. I sort the combined alphabet after making it a set, so that, when printing the cipher, users can easily find which character maps to where (and which characters don't map) by going through alphabetically, instead of searching an arbitrary ordering of characters.

For powers of substitution ciphers (again, using `__pow__`), I validate the exponent as I did for `CaesarCipher`, before simply re-encoding the cipher's `permutation` `other - 1` times (for a total of `other` encodings of alphabet). And then I simply return the new `SubstitutionCipher`, with its original alphabet. This is more efficient than composing it with itself `other - 1` times, because `__matmul__` has to do things like combining and sorting the alphabet each time, which would be redundant here.

## Task B2: Operations on Affine Ciphers &nbsp;&nbsp; [2 marks]

In [7]:
# [Demonstrate]

# definitions:

def fast_sum_powers(a, n: int):
    """Computes a^n + a^{n-1} + ... + a + 1"""
    if not isinstance(n, int):
        raise TypeError("n must be an int")
    if not isinstance(a, int):
        raise TypeError("a must be a numpy array")
    if n == 0:
        return 1
    result = a
    for _ in range(n - 1):
        result = a * result + a
    return result + 1

# adding more overloads to AffineCipher:
class AffineCipher(SubstitutionCipher):
    """[Placeholder]"""
    
    def __init__(self, a: int, b: int, alphabet: str = default_alphabet):
        if not isinstance(a, int):
            raise TypeError("a must be an int")
        if not isinstance(b, int):
            raise TypeError("b must be an int")
        l = len(alphabet)
        if sp.gcd(a, l) != 1:
            raise ValueError("a and len(alphabet) must be coprime (otherwise the cipher isn't decodable)")
        self.alphabet = alphabet # needed for the permutation calculation
        self.alphabet_char_indices = {char: i for (i, char) in enumerate(alphabet)} # needed for the permutation calculation
        self.a = a % l # keeps values in interval [0, l), for consistency when comparing ciphers
        self.b = b % l # (as above)
        super().__init__(self._get_affine_permutation(), alphabet)


    def __str__(self) -> str:
        return f"Affine cipher (i ↦ {self.a}i + {self.b}, alphabet: {self.alphabet})"
    
    # Helper methods:

    def _encode_alphabet_char_affine(self, char: str):
        # doesn't need to check for char's type, because this function is only used on alphabets
        # doesn't need to support uppercase letters, because it'll only be used on alphabets
        old_index = self.alphabet_char_indices.get(char)
        # doesn't need to support letters not in the alphabet, since it'll only be used on alphabets 
        new_index = (self.a * old_index + self.b) % len(self.alphabet) # mod prevents index underflows and overflows
        return self.alphabet[new_index]

    def _get_affine_permutation(self):
        return "".join(self._encode_alphabet_char_affine(char) for char in self.alphabet) # encode the alphabet to get the permutation

    # Operator overloads:

    def __add__(self, other) -> "AffineCipher":
        if not isinstance(other, AffineCipher):
            raise TypeError(f"Can only add (+) AffineCipher to AffineCipher")
        if self.a != other.a:
            raise ValueError("Operands must have the same multiplier `a`")
        if self.alphabet != other.alphabet:
            raise ValueError("Operands must have the same alphabet")
        return AffineCipher(self.a, self.b + other.b, self.alphabet)

    def __neg__(self) -> "AffineCipher":
        return AffineCipher(self.a, -self.b, self.alphabet)
        
    def __sub__(self, other) -> "AffineCipher":
        if not isinstance(other, AffineCipher):
            raise TypeError(f"Can only subtract (-) AffineCipher from AffineCipher")
        # other errors are handled by __add__ already
        return self + (-other)
        
    def __mul__(self, other) -> "AffineCipher":
        if not isinstance(other, int):
            raise TypeError(f"Can only multiply (*) AffineCipher by int")
        return AffineCipher(self.a, self.b * other, self.alphabet)

    def __rmul__(self, other) -> "AffineCipher":
        return self * other

    def __matmul__(self, other) -> SubstitutionCipher:
        # error handling is handled by SubstitutionCipher.__matmul__
        if isinstance(other, AffineCipher) and (self.alphabet == other.alphabet):
            # A_c_d ∘ A_a_b = A_{ac}_{bc+d}, because c(ai+b)+d = (ac)i+(bc+d) (mod len(alphabet))
            # (where self = A_c_d and other = A_a_b)
            a_new = other.a * self.a
            b_new = other.b * self.a + self.b
            return AffineCipher(a_new, b_new, self.alphabet)
        if isinstance(other, CaesarCipher) and (self.alphabet == other.alphabet): # treat CaesarCipher as AffineCipher
            # other.a = 1, other.b = other.shift
            a_new = self.a
            b_new = other.shift * self.a + self.b
            return AffineCipher(a_new, b_new, self.alphabet)
        return SubstitutionCipher.__matmul__(self, other) # use SubstitutionCipher's @ instead
        
    def __pow__(self, other) -> SubstitutionCipher:
        if not isinstance(other, int):
            raise TypeError(f"Can only raise (**) AffineCipher to the power of an int")
        if other < 0:
            raise ValueError(f"Can only raise (**) AffineCipher to a non-negative power")
        if other == 0:
            return SubstitutionCipher() # identity "cipher"
        # (A_a_b)^n = A_{a^n}_{b(Σ_{i=0}^{n-1} a^i)} [see Justification below]
        a_new = self.a ** other
        b_new = self.b * fast_sum_powers(self.a, other - 1)
        return AffineCipher(a_new, b_new, self.alphabet)
        

# encoding and decoding the six secret messages:

A_5_3_qwerty = AffineCipher(5, 3, qwerty)
A_7_2_qwerty = AffineCipher(7, 2, qwerty)
A_5_2_ambidextrous = AffineCipher(5, 2, ambidextrous)
A_3_2 = AffineCipher(3, 2)
A_4_2_vowels = AffineCipher(4, 2, vowels)
A_4_1_vowels = AffineCipher(4, 1, vowels)
A_5_3_qwerty = AffineCipher(5, 3, qwerty)
A_8_2_consonants = AffineCipher(8, 2, consonants)

cipher1 = A_5_3_qwerty @ A_7_2_qwerty
cipher2 = A_5_2_ambidextrous ** 4
cipher3 = 5 * A_3_2
cipher4 = -A_3_2
cipher5 = A_4_2_vowels + A_4_1_vowels
cipher6 = A_5_3_qwerty @ A_8_2_consonants
ciphers = [cipher1, cipher2, cipher3, cipher4, cipher5, cipher6]

for cipher in ciphers:
    print(border)
    print(cipher)
    print(border)
    cipher.print_encryption_decryption(default_alphabet)
    for message in messages:
        cipher.print_encryption_decryption(message)
        print(border)
    print() # spacing


Affine cipher (i ↦ 9i + 13, alphabet: qwertyuiopasdfghjklzxcvbnm)
Uncoded message: abcdefghijklmnopqrstuvwxyz
Encoded message: mdxkyqplnwaztcijfgobhrvsue
Decoded message: abcdefghijklmnopqrstuvwxyz
Uncoded message: Agent 007: "Shaken, not stirred!"
Encoded message: Mpycb 007: "Olmayc, cib obnggyk!"
Decoded message: Agent 007: "Shaken, not stirred!"
Uncoded message: [At 221B Baker Street] Holmes (to Watson): "Elementary, my dear Watson!"
Encoded message: [Mb 221D Dmayg Obgyyb] Liztyo (bi Vmboic): "Yzytycbmgu, tu kymg Vmboic!"
Decoded message: [At 221B Baker Street] Holmes (to Watson): "Elementary, my dear Watson!"
Uncoded message: Why so serious?! Let's put a smile on that face.
Encoded message: Vlu oi oygniho?! Zyb'o jhb m otnzy ic blmb qmxy.
Decoded message: Why so serious?! Let's put a smile on that face.
Uncoded message: In 1950, Turing asked: "Can machines think?"
Encoded message: Nc 1950, Bhgncp moayk: "Xmc tmxlncyo blnca?"
Decoded message: In 1950, Turing asked: "Can machines thi

## **[Justify]**
I overloaded `AffineCipher`'s relevant operators and I validated inputs exactly how I did in Task B2, but with the specification's definitions for affine cipher operations instead of for Caesar cipher ones.

For $A_{a,b,\sigma}$ to be a valid cipher, $a$ and the length of $\sigma$ must be coprime, otherwise it can't be decoded, as discussed in Task A2 (but with $\sigma = $`alphabet)`).

For compositions with `__matmul__`, if both are affine ciphers on the same alphabet, the composed cipher has values `a = other.a * self.a` and `b = other.b * self.a + self.b`. This is because, for two ciphers $A_{a,b}$ and $A_{c,d}$, their composition, $A_{a,b} \circ A_{c,d}$, represents, for each index $i$, the map $i \mapsto a(ci+d)+b = (ac)i+(ad+b)$, so $A_{a,b} \circ A_{c,d} = A_{ac,ad+b}$. And, if the other has a different alphabet or is a different cipher type, I fall back onto `SubstitutionCipher.__matmul__`, as I explained in Task B1. This produces a valid affine cipher, because both $a$ and $c$ are coprime to `len(alphabet)`, so $ac$ must be as well.

Also, my code treats affine-Caesar compositions the same way it treats affine-affine, simply using the relationship $C_c = A_{1,c}$. This way, the core identity of the cipher composition being (essentially) affine is preserved.

For powers (using `__pow__`), I derived a formula as follows. Let the map $\phi: i \mapsto ai+b$ represent the affine cipher $A_{a,b}$. If we repeatedly apply $\phi$, we get
$$
\begin{align}
\phi(i) &= ai+b \\
\phi^2(i) &= a(ai + b) + b = a^2 i + ab + b \\
\phi^3(i) &= a(a^2 i + ab + b) + b = a^3 i + a^2 b + ab + b \\
&\vdotswithin{=} \\ 
\phi^n(i) &= a^n i + (a^{n-1} + a^{n-2} + \dots + a + 1)b \\
&= a^n i + b \sum_{i=0}^{n-1} a^i.
\end{align}
$$

I could simplify further using the geometric series theorem, but that would involve a $(a-1)^{-1}$ term, which doesn't necessarily exist modulo `len(alphabet)` (even if $a^{-1}$ is required to), so the calculation wouldn't always be valid. Instead, I defined `fast_sum_powers()`, which uses Horner's method (as explained in semester 1) to efficiently compute a sum of powers. This is because $\sum_{i=0}^{n-1} a^i = a(\dots(a(a(a+1)+1)+1)\dots)+1$, which requires less operations than simply $a^n + a^{n-1} + \dots + a + 1$. This means that I can define the composition's values as `a = self.a ** other` and `b = self.b * fast_sum_powers(self.a, other - 1)`, and return the new affine cipher. This produces a valid affine cipher, because $a$ is coprime to `len(alphabet)`, so $a^n$ must be as well.

## Task B3: Compositions on Substitution Ciphers &nbsp;&nbsp; [2 marks]

In [8]:
# [Demonstrate]

# encoding and decoding the six secret messages:

S_reverse = SubstitutionCipher(reverse)
S_split = SubstitutionCipher(split)
S_reverse_qwerty = SubstitutionCipher(reverse, qwerty)
S_split_vowels = SubstitutionCipher(split, vowels)
S_split_ambidextrous = SubstitutionCipher(split, ambidextrous)

cipher1 = S_reverse @ S_split
cipher2 = S_reverse_qwerty @ S_split_vowels
cipher3 = S_split_ambidextrous ** 3
ciphers = [cipher1, cipher2, cipher3]

for cipher in ciphers:
    print(border)
    print(cipher)
    print(border)
    
    for message in messages:
        cipher.print_encryption_decryption(message)
        print(border)
    
    print() # spacing

print("S_reverse ** 2: ", S_reverse ** 2)
print("Identity cipher: ", SubstitutionCipher(default_alphabet, default_alphabet))

Substitution cipher (abcdefghijklmnopqrstuvwxyz ↦ zxvtrpnljhfdbacegikmoqsuwy)
Uncoded message: Agent 007: "Shaken, not stirred!"
Encoded message: Znram 007: "Klzfra, acm kmjiirt!"
Decoded message: Agent 007: "Shaken, not stirred!"
Uncoded message: [At 221B Baker Street] Holmes (to Watson): "Elementary, my dear Watson!"
Encoded message: [Zm 221X Xzfri Kmirrm] Lcdbrk (mc Szmkca): "Rdrbramziw, bw trzi Szmkca!"
Decoded message: [At 221B Baker Street] Holmes (to Watson): "Elementary, my dear Watson!"
Uncoded message: Why so serious?! Let's put a smile on that face.
Encoded message: Slw kc krijcok?! Drm'k eom z kbjdr ca mlzm pzvr.
Decoded message: Why so serious?! Let's put a smile on that face.
Uncoded message: In 1950, Turing asked: "Can machines think?"
Encoded message: Ja 1950, Moijan zkfrt: "Vza bzvljark mljaf?"
Decoded message: In 1950, Turing asked: "Can machines think?"
Uncoded message: [Act 3, Scene 1] To be, or not to be?
Encoded message: [Zvm 3, Kvrar 1] Mc xr, ci acm mc xr?
Decod

**[Justify]**
My design choices for `SubstitutionCipher` have already been justified in Tasks A1 and A3; nothing has changed.

For any string (and hence any alphabet), reversing it twice returns the original string. So, since $S_{\texttt{reverse}}^2$'s `permutation` is given by reversing `default` twice, its permutation is the same as its alphabet. So $S_{\texttt{reverse}}^2$ is the identity cipher, which maps all charactres to themselves. (See the last two outputs above for a comparison.)

## Task B4: Mixed Combination &nbsp;&nbsp; [2 marks]

In [9]:
# [Demonstrate]

a = -7
b = 13
c = 2
a1 = 3
b1 = -3
a2 = -7
b2 = 3
sigma = ambidextrous
alpha = consonants
beta = vowels
gamma = qwerty
n = 4

C_a_sigma = CaesarCipher(a, sigma)
C_b_sigma = CaesarCipher(b, sigma)
C_c_sigma = CaesarCipher(c, sigma)
S_split_alpha = SubstitutionCipher(split, alpha)
A_a1_b1_beta = AffineCipher(a1, b1, beta)
A_a2_b2_gamma = AffineCipher(a2, b2, gamma)

cipher = (((C_a_sigma + 4 * C_b_sigma - 2 * C_c_sigma) @ (S_split_alpha ** 2)) ** 2) @ ((n * A_a1_b1_beta) @ A_a2_b2_gamma)

print(border)
print(cipher)
print(border)

for message in messages:
    cipher.print_encryption_decryption(message)
    print(border)

Substitution cipher (abcdefghijklmnopqrstuvwxyz ↦ egxyodqiuvsrthznfpjlkwabcm)
Uncoded message: Agent 007: "Shaken, not stirred!"
Encoded message: Eqohl 007: "Jiesoh, hzl jluppoy!"
Decoded message: Agent 007: "Shaken, not stirred!"
Uncoded message: [At 221B Baker Street] Holmes (to Watson): "Elementary, my dear Watson!"
Encoded message: [El 221G Gesop Jlpool] Izrtoj (lz Aeljzh): "Orotohlepc, tc yoep Aeljzh!"
Decoded message: [At 221B Baker Street] Holmes (to Watson): "Elementary, my dear Watson!"
Uncoded message: Why so serious?! Let's put a smile on that face.
Encoded message: Aic jz jopuzkj?! Rol'j nkl e jturo zh liel dexo.
Decoded message: Why so serious?! Let's put a smile on that face.
Uncoded message: In 1950, Turing asked: "Can machines think?"
Encoded message: Uh 1950, Lkpuhq ejsoy: "Xeh texiuhoj liuhs?"
Decoded message: In 1950, Turing asked: "Can machines think?"
Uncoded message: [Act 3, Scene 1] To be, or not to be?
Encoded message: [Exl 3, Jxoho 1] Lz go, zp hzl lz go?
Decod

## **[Justify]**
I chose the following values:
- $a = -7$
- $b = 13$
- $c = 2$
- $a_1 = 3$
- $b_1 = -3$
- $a_2 = -7$
- $b_2 = 3$
- $\sigma =$ `ambidextrous`
- $\alpha =$ `consonants`
- $\beta =$ `vowels`
- $\gamma =$ `qwerty`
- $n = 4$

I had the following restrictions (besides those specified by the task):
- all numbers had to be integers
- all alphabets had to satisfy `is_valid_alphabet()`, as justified in Task 1A
- $a_1$ had to be coprime with the length of $\beta$
- $a_2$ had to be coprime with the length of $\gamma$

My code evaluates the cipher as `(((C_a_sigma + 4 * C_b_sigma - 2 * C_c_sigma) @ (S_split_alpha ** 2)) ** 2) @ ((n * A_a1_b1_beta) @ A_a2_b2_gamma)`, using the operator overloads defined throughout Tasks B1 and B2.

`CaesarCipher`'s `__add__`, `__rmul__`, and `__sub__` methods handle the $C_{a,\sigma} + 4C_{b,\sigma} - 2C_{c,\sigma}$ part, evaluating it as $C_{a + 4b -2c}$. `SubstitutionCipher.__pow__` squares $S_{\text{split},\alpha}$. Since the cipher types differ, $C_{a + 4b -2c} \cdot (S_{\text{split},\alpha})^2$ falls back to `SubstitutionCipher.__matmul__`, creating a new `SubstitutionCipher` with an alphabet that merges $\sigma$ and $\alpha$ (namely `default`, because `ambidextrous` has every vowel, and it's being merged with `consonants`), and its permutation (the result of encoding the new alphabet once with each cipher), `"etgluqvzrwmxhcbfkpayisdnjo"`. This is then squared (again, with `SubstitutionCipher.__pow__`).

`AffineCipher`'s `__rmul__` evaluates $n A_{a_1,b_1,\beta} = A_{a_1,nb_1,\beta}$. Since the two affine ciphers have different alphabets, `__matmul__` falls back to `SubstitutionCipher.__matmul__`, which merges the two alphabets into one (again, `default`, because `qwerty` has every character in `default`, and it's sorted alphabetically), with permutation `"sndbhjpeacguokmlritwfyvzxq"`.

These two ciphers are then composed again with `SubstitutionCipher.__matmul__`, yielding a substitution cipher with `alphabet = default` and `permutation = "egxyodqiuvsrthznfpjlkwabcm"`.

---
# Part C: Affine Hill Cipher &nbsp;&nbsp; [4 marks]

## Task C1: Affine Hill Cipher &nbsp;&nbsp; [2 marks]

In [10]:
# [Demonstrate]

# definitions:
class AffineHillCipher(Cipher):
    """[Placeholder]"""
    
    def __init__(self, A: sp.Matrix, b: sp.Matrix, alphabet: str = default_alphabet):
        if not isinstance(A, sp.Matrix):
            raise TypeError("A must be a SymPy Matrix")
        if not isinstance(b, sp.Matrix):
            raise TypeError("b must be a SymPy Matrix")
        if b.shape[1] != 1:
            raise ValueError("A must be n-by-n and b must have n dimensions")
        self.block_size = A.shape[0]
        if b.shape[0] != self.block_size:
            raise ValueError("b must be a column vector (n-by-1 SymPy Matrix)")
        if self.block_size != A.shape[1]:
            raise ValueError("A must be square")
        super().__init__(alphabet)
        self.A_inv = A.inv_mod(len(self.alphabet))
        self.A = A
        self.b = b

    def __str__(self):
        return f"Affine Hill Cipher: (\nA: {self.A}, \nb: {self.b}, \nalphabet: {self.alphabet})"

    def encode(self, msg: str) -> str:
        blocks, block_char_indices_in_msg = self._get_blocks_and_block_char_indices_in_msg(msg)
        flattened_mapped_blocks = self._blocks_to_flattened_mapped_blocks(blocks, encode=True)
        return self._get_mapped_msg_from_blocks(msg, flattened_mapped_blocks, block_char_indices_in_msg)

    def decode(self, msg: str) -> str:
        blocks, block_char_indices_in_msg = self._get_blocks_and_block_char_indices_in_msg(msg)
        flattened_mapped_blocks = self._blocks_to_flattened_mapped_blocks(blocks, encode=False)
        return self._get_mapped_msg_from_blocks(msg, flattened_mapped_blocks, block_char_indices_in_msg)

    # Helper functions:
        
    def _get_blocks_and_block_char_indices_in_msg(self, msg: str) -> tuple[list[list[str]], list[int]]:
        if not isinstance(msg, str):
            raise TypeError("msg must be a str")
        blocks = []
        block_char_indices_in_msg = []
        current_block = []
        current_block_length = 0
        for i in range(len(msg)):
            lower_msg = msg[i].lower()
            if lower_msg in self.alphabet: # if we can encode it,
                current_block.append(lower_msg) # add it to the current block
                block_char_indices_in_msg.append(i) # add its index in msg to the list of indices
                current_block_length += 1
            if current_block_length == self.block_size: # if we've got a completed block,
                blocks.append(current_block) # add to blocks
                # reset current block variables:
                current_block = []
                current_block_length = 0
        if current_block_length > 0: # if there's an incomplete last block
            block_char_indices_in_msg = block_char_indices_in_msg[:-current_block_length] # trim block_char_indices_in_msg to not include the indices for the incomplete last block
        return blocks, block_char_indices_in_msg
    
    def _map_block(self, block: list[str], encode: bool) -> list[str]:
        block_char_indices = sp.Matrix([self.alphabet_char_indices.get(char) for char in block]) # get indices as numpy array
        if encode == True:
            mapped_block_char_indices = ((self.A * block_char_indices) + self.b) % len(self.alphabet) # map indices
        else:
            mapped_block_char_indices = (self.A_inv * (block_char_indices - self.b)) % len(self.alphabet) # map indices
        return [self.alphabet[index] for index in list(mapped_block_char_indices)] # turn indices back into characters
        
    def _blocks_to_flattened_mapped_blocks(self, blocks: list[list[str]], encode: bool) -> list[str]:
        return [char
                for block in blocks
                for char in self._map_block(block, encode)] # flattens the list, while encoding each block
        
    def _get_mapped_msg_from_blocks(self, msg: str, flattened_mapped_blocks: list[str], block_char_indices_in_msg: list[int]) -> str:
        msg_list = list(msg)
        for i in range(len(block_char_indices_in_msg)):
            msg_i = block_char_indices_in_msg[i]
            if msg[msg_i] == msg[msg_i].upper(): # if original was uppercase 
                msg_list[msg_i] = flattened_mapped_blocks[i].upper()
            else:
                msg_list[msg_i] = flattened_mapped_blocks[i] # the elements of flattened_mapped_blocks are always lowercase, so there's no need to use .lower()
        return "".join(msg_list)


# encoding and decoding the six secret messages:

A = sp.Matrix([[2, 1], [1, 1]])
b = sp.Matrix([1, 0])
cipher = AffineHillCipher(A, b, vowels)
print(border)
print(cipher)
print(border)
for message in messages:
    cipher.print_encryption_decryption(message)
    print(border)


Affine Hill Cipher: (
A: Matrix([[2, 1], [1, 1]]), 
b: Matrix([[1], [0]]), 
alphabet: aeiou)
Uncoded message: Agent 007: "Shaken, not stirred!"
Encoded message: Igent 007: "Shiken, nut starred!"
Decoded message: Agent 007: "Shaken, not stirred!"
Uncoded message: [At 221B Baker Street] Holmes (to Watson): "Elementary, my dear Watson!"
Encoded message: [Et 221B Bakur Striet] Hulmes (tu Wutson): "Ulimontery, my doer Wutson!"
Decoded message: [At 221B Baker Street] Holmes (to Watson): "Elementary, my dear Watson!"
Uncoded message: Why so serious?! Let's put a smile on that face.
Encoded message: Why so suroaas?! Lat's put u smelo in thot fice.
Decoded message: Why so serious?! Let's put a smile on that face.
Uncoded message: In 1950, Turing asked: "Can machines think?"
Encoded message: Un 1950, Terang iskod: "Cen mochinas thonk?"
Decoded message: In 1950, Turing asked: "Can machines think?"
Uncoded message: [Act 3, Scene 1] To be, or not to be?
Encoded message: [Ict 3, Scene 1] Tu be, ur n

## **[Justify]**
`AffineHillCipher` inherits from Task 1A's `Cipher` (instead of `SubstitutionCipher`) because it operates on blocks of characters, so `SubstitutionCipher` single-character-encoding logic doesn't work. But it still defines `encode()`, `decode()`, and `__str__()`.

To encode or **decode** a message, I split the ciphertext parts of it into blocks of character indices (leaving out the last block if it's unfinished) and keep track of their indices within the original message, using `_get_blocks_and_block_char_indices_in_msg()`. My helper methods for `encode()` and `decode()` share the logic for both directions of encryption, because of the same principles discussed in Task 1A (and using the same conventions for `_` in method names). 

`_map_block()` maps the character index vector $\mathbf{v}$ to $A \mathbf{v} + \mathbf{b} \ \text{mod}$ `len(alphabet)` when encoding (or to $A^{-1}(\mathbf{v} - \mathbf{b}) \ \text{mod}$ `len(alphabet)` when decoding). It uses a list comprehension to turn the vector back into a list, while turning each index back into its corresponding character in `alphabet` and returns the result.

`_blocks_to_flattened_mapped_blocks()` uses a nested list comprehension on the list of blocks to flatten it into a 1d list, while encoding or decoding each block, for efficiency.

`_get_mapped_msg_from_blocks()` takes in the flattened list of mapped blocks from the above method, and uses the list of indices in `msg` of the block characters (generated by `_get_blocks_and_block_char_indices_in_msg()`) to change the ciphertext characters in `msg` into their mapped characters (checking the original string to preserve the original capitalisation). It needs to first convert `msg` into a list though, since indexing is required. After it's finished mapping, it returns a string version of the message, using `"".join`.

To decode the cipher, my code applies the map $v \mapsto A^{-1}(\mathbf{v} - \mathbf{b}) \ \text{mod}$ `len(alphabet)`, because that's the inverse of the encoding map $v \mapsto A\mathbf{v} + \mathbf{b} \ \text{mod}$ `len(alphabet)`. This requires $A$ to be invertible, modulo `len(alphabet)`, which in turn requires $\det(A)$ to be coprime with `len(alphabet)` (because the determinant has to be invertible for the matrix to be). 

In the constructor, while I validate inputs (and raising errors) to ensure that $A$ is a square matrix and that $\mathbf{b}$ is a vector of the right length, I don't need to test if $A$ is invertible myself. This is because `sp.inv_mod()` raises an error itself if $A$ isn't invertible modulo `len(alphabet)`, meaning any validation I do for that would be redundant.

To construct a different working example, all one needs is a matrix which is invertible, modulo the length of their chosen alphabet, and a suitable vector to go with it. For example, with the `vowels` alphabet (of length $5$), let
$$A = \begin{pmatrix} 2 & 4 & 2 \\ 1 & 3 & 3 \\ 3 & 2 & 1 \end{pmatrix},
\qquad \mathbf{b} = \begin{pmatrix} 3 \\ 2 \\ 5 \end{pmatrix}.$$
$\det(A)=2 \ \text{mod} \ 5$, so the matrix is valid. And $A$'s number of rows (or columns) is equal to $\mathbf{b}$'s number of rows. So the example is valid (with a block size of 3).

## Task C2: Operations on Affine Hill Ciphers &nbsp;&nbsp; [2 marks]

In [11]:
# [Demonstrate]

# definitions:
def fast_sum_matrix_powers(A: sp.Matrix, n: int):
    """Computes A^n + A^{n-1} + ... + A + I"""
    if not isinstance(n, int):
        raise TypeError("n must be an int")
    if not isinstance(A, sp.Matrix):
        raise TypeError("A must be a SymPy Matrix")
    order = A.shape[0]
    if order != A.shape[1]:
        raise ValueError("A must be square") 
    if n == 0:
        return sp.eye(order)
    result = A
    for _ in range(n - 1):
        result = A @ result + A
    return result + sp.eye(order)


class CipherComposition(Cipher):
    """[Placeholder]"""

    def __init__(self, second: Cipher, first: Cipher):
        if not isinstance(second, Cipher):
            raise TypeError("second must be a Cipher")
        if not isinstance(first, Cipher):
            raise TypeError("first must be a Cipher")
        self.second = second
        self.first = first
        
    def __str__(self) -> str:
        return ("Composition Cipher of second∘first, with\n"
                f"first = {{ {self.first} }}, and\n"
                f"second = {{ {self.second} }}")
            
    def encode(self, msg: str) -> str:
        # error handling is taken care of by the other encode functions
        return self.second.encode(self.first.encode(msg))
        
    def decode(self, msg: str) -> str:
        # error handling is taken care of by the other decode functions
        return self.first.decode(self.second.decode(msg))

    def __matmul__(self, other: Cipher) -> "CipherComposition":
        if not isinstance(other, Cipher):
            raise TypeError(f"Can only compose (@) CipherComposition with Cipher")
        return CipherComposition(self, other)

# adding overloads to AffineHillCipher:
class AffineHillCipher(Cipher):
    """[Placeholder]"""
    
    def __init__(self, A: sp.Matrix, b: sp.Matrix, alphabet: str = default_alphabet):
        if not isinstance(A, sp.Matrix):
            raise TypeError("A must be a SymPy Matrix")
        if not isinstance(b, sp.Matrix):
            raise TypeError("b must be a SymPy Matrix")
        if b.shape[1] != 1:
            raise ValueError("A must be n-by-n and b must have n dimensions")
        self.block_size = A.shape[0]
        if b.shape[0] != self.block_size:
            raise ValueError("b must be a column vector (n-by-1 SymPy Matrix)")
        if self.block_size != A.shape[1]:
            raise ValueError("A must be square")
        super().__init__(alphabet)
        self.A_inv = A.inv_mod(len(self.alphabet))
        self.A = A
        self.b = b

    def __str__(self) -> str:
        return f"Affine Hill Cipher: (\nA: {self.A}, \nb: {self.b}, \nalphabet: {self.alphabet})"

    def encode(self, msg: str) -> str:
        blocks, block_char_indices_in_msg = self._get_blocks_and_block_char_indices_in_msg(msg)
        flattened_mapped_blocks = self._blocks_to_flattened_mapped_blocks(blocks, encode=True)
        return self._get_mapped_msg_from_blocks(msg, flattened_mapped_blocks, block_char_indices_in_msg)

    def decode(self, msg: str) -> str:
        blocks, block_char_indices_in_msg = self._get_blocks_and_block_char_indices_in_msg(msg)
        flattened_mapped_blocks = self._blocks_to_flattened_mapped_blocks(blocks, encode=False)
        return self._get_mapped_msg_from_blocks(msg, flattened_mapped_blocks, block_char_indices_in_msg)

    # Helper functions:
        
    def _get_blocks_and_block_char_indices_in_msg(self, msg: str) -> tuple[list[list[str]], list[int]]:
        if not isinstance(msg, str):
            raise TypeError("msg must be a str")
        blocks = []
        block_char_indices_in_msg = []
        current_block = []
        current_block_length = 0
        for i in range(len(msg)):
            lower_msg = msg[i].lower()
            if lower_msg in self.alphabet: # if we can encode it,
                current_block.append(lower_msg) # add it to the current block
                block_char_indices_in_msg.append(i) # add its index in msg to the list of indices
                current_block_length += 1
            if current_block_length == self.block_size: # if we've got a completed block,
                blocks.append(current_block) # add to blocks
                # reset current block variables:
                current_block = []
                current_block_length = 0
        if current_block_length > 0: # if there's an incomplete last block
            block_char_indices_in_msg = block_char_indices_in_msg[:-current_block_length] # trim block_char_indices_in_msg to not include the indices for the incomplete last block
        return blocks, block_char_indices_in_msg
    
    def _map_block(self, block: list[str], encode: bool) -> list[str]:
        block_char_indices = sp.Matrix([self.alphabet_char_indices.get(char) for char in block]) # get indices as numpy array
        if encode == True:
            mapped_block_char_indices = ((self.A * block_char_indices) + self.b) % len(self.alphabet) # map indices
        else:
            mapped_block_char_indices = (self.A_inv * (block_char_indices - self.b)) % len(self.alphabet) # map indices
        return [self.alphabet[index] for index in list(mapped_block_char_indices)] # turn indices back into characters
        
    def _blocks_to_flattened_mapped_blocks(self, blocks: list[list[str]], encode: bool) -> list[str]:
        return [char
                for block in blocks
                for char in self._map_block(block, encode)] # flattens the list, while encoding each block
        
    def _get_mapped_msg_from_blocks(self, msg: str, flattened_mapped_blocks: list[str], block_char_indices_in_msg: list[int]) -> str:
        msg_list = list(msg)
        for i in range(len(block_char_indices_in_msg)):
            msg_i = block_char_indices_in_msg[i]
            if msg[msg_i] == msg[msg_i].upper(): # if original was uppercase 
                msg_list[msg_i] = flattened_mapped_blocks[i].upper()
            else:
                msg_list[msg_i] = flattened_mapped_blocks[i] # the elements of flattened_mapped_blocks are always lowercase, so there's no need to use .lower()
        return "".join(msg_list)

    # Operator overloads:
    
    def __add__(self, other: "AffineHillCipher") -> "AffineHillCipher":
        if not isinstance(other, AffineHillCipher):
            raise TypeError(f"Can only add (+) AffineHillCipher to AffineHillCipher")
        if self.A != other.A:
            raise ValueError("Operands must have the same multiplier `A`")
        if self.alphabet != other.alphabet:
            raise ValueError("Operands must have the same alphabet")
        return AffineHillCipher(self.A, self.b + other.b, self.alphabet)

    def __neg__(self) -> "AffineHillCipher":
        return AffineHillCipher(self.A, -self.b, self.alphabet)
        
    def __sub__(self, other: "AffineHillCipher") -> "AffineHillCipher":
        if not isinstance(other, AffineHillCipher):
            raise TypeError(f"Can only subtract (-) AffineHillCipher from AffineHillCipher")
        # other errors are handled by __add__ already
        return self + (-other)
        
    def __mul__(self, other: int) -> "AffineHillCipher":
        if not isinstance(other, int):
            raise TypeError(f"Can only multiply (*) AffineHillCipher by int")
        return AffineHillCipher(self.A, self.b * other, self.alphabet)

    def __rmul__(self, other: int) -> "AffineHillCipher":
        return self * other

    def __matmul__(self, other: Cipher) -> Cipher:
        if not isinstance(other, AffineHillCipher):
            raise TypeError(f"Can only compose (@) AffineHillCipher with Cipher")
        
        if isinstance(other, AffineHillCipher) and (self.block_size == other.block_size) and (self.alphabet == other.alphabet):
            # H_C_d ∘ H_A_b = H_{CA}_{Cb+d}, because C(Ai+b)+d = (AC)i+(Cb+d) (mod len(alphabet))
            # (where self = H_C_d and other = H_A_b)
            A_new = self.A * other.A
            b_new = self.A * other.b + self.b
            return AffineHillCipher(A_new, b_new, self.alphabet)
        return CipherComposition(self, other)

    def __pow__(self, other: int) -> "AffineHillCipher":
        if not isinstance(other, int):
            raise TypeError(f"Can only raise (**) AffineHillCipher to the power of an int")
        if other < 0:
            raise ValueError(f"Can only raise (**) AffineHillCipher to a non-negative power")
        if other == 0: # need to return identity cipher
            A_new = sp.eye(self.block_size)
            b_new = sp.zeros(self.block_size)
        else:
            A_new = self.A ** other
            b_new = fast_sum_matrix_powers(self.A, other - 1) * self.b
        return AffineHillCipher(A_new, b_new, self.alphabet)

        
# encoding and decoding the six secret messages:

A2 = sp.Matrix([[2, 4, 2], [1, 3, 3], [3, 2, 1]])
b2 = sp.Matrix([3, 2, 0])
b_prime = sp.Matrix([4, 3])

H_A_b_vowels = AffineHillCipher(A, b, vowels)
H_A2_b2_vowels = AffineHillCipher(A2, b2, vowels)
H_A_b_vowels = AffineHillCipher(A, b, vowels)
H_A_b_prime_vowels = AffineHillCipher(A, b_prime, vowels)

cipher1 = H_A_b_vowels @ H_A2_b2_vowels
cipher2 = H_A_b_vowels ** 3
cipher3 = -H_A_b_vowels
cipher4 = H_A_b_vowels + H_A_b_prime_vowels
ciphers = [cipher1, cipher2, cipher3, cipher4]


for cipher in ciphers:
    print(border)
    print(cipher)
    print(border)
    
    for message in messages:
        cipher.print_encryption_decryption(message)
        print(border)
    
    print() # spacing
    

Composition Cipher of second∘first, with
first = { Affine Hill Cipher: (
A: Matrix([[2, 4, 2], [1, 3, 3], [3, 2, 1]]), 
b: Matrix([[3], [2], [0]]), 
alphabet: aeiou) }, and
second = { Affine Hill Cipher: (
A: Matrix([[2, 1], [1, 1]]), 
b: Matrix([[1], [0]]), 
alphabet: aeiou) }
Uncoded message: Agent 007: "Shaken, not stirred!"
Encoded message: Agint 007: "Shekon, not sturred!"
Decoded message: Agent 007: "Shaken, not stirred!"
Uncoded message: [At 221B Baker Street] Holmes (to Watson): "Elementary, my dear Watson!"
Encoded message: [Et 221B Bakor Streut] Holmis (tu Wutsun): "Alomentory, my door Wetsun!"
Decoded message: [At 221B Baker Street] Holmes (to Watson): "Elementary, my dear Watson!"
Uncoded message: Why so serious?! Let's put a smile on that face.
Encoded message: Why su seruaus?! Lot's pot i smele un thet fice.
Decoded message: Why so serious?! Let's put a smile on that face.
Uncoded message: In 1950, Turing asked: "Can machines think?"
Encoded message: In 1950, Turang oskod

## **[Justify]**
I overloaded `AffineHillCipher`'s relevant operators and validated inputs in the same way as I did for Task B2, but just swapping `a` with `A`, to represent the matrix/vector versions of the operations. This also meant that I needed to define `fast_sum_matrix_powers()`, to deal with the geometric series with a matrix instead of a scalar; otherwise, it has the same logic as its version in Task B2's `fast_sum_powers()`.

However, for compositions of affine Hill ciphers where the value of `alphabet` (or compositions with different ciphers), I couldn't rely on `SubstitutionCipher`'s alphabet merging method, because affine Hill ciphers don't just substitute with a permutation, as `SubstitutionCipher`s do. So I defined a `CipherComposition` class (inheriting from `Cipher`).

`CipherComposition` has two attributes, `first` and `second`, representing the first cipher to be applied in the composition, and the second, repectively. The constructor also validates that these are both `Cipher` objects, raising an error if not. The `encode()` method simply uses `first.encode()` and then `second.encode()` on the message, to encode it as it should be. And `decode()` works similarly, but it has to undo the encoding, so it uses `second.decode()` first, and then `first.decode()`.

`CipherComposition` can also be composed with other ciphers (including `SubstitutionCiphers`) by defining a new `CipherComposition` object, with `second = self` and `first = other`. 

With that defined, `AffineHillCipher`'s `__matmul__` method can then deal with the rest of the composition cases by returning `CipherComposition(self, other)`.

Also, for powers of`AffineHillCipher`, I had to use `sp.eye()` and `sp.zeros()` to define an identity affine Hill cipher, because I couldn't use `SubstitutionCipher()`, as explained above.

I chose $A_2$ with the requirements I gave in Task C1. In fact, it's the same matrix I gave in my example for that Task's Justify, namely
$$A = \begin{pmatrix} 2 & 4 & 2 \\ 1 & 3 & 3 \\ 3 & 2 & 1 \end{pmatrix}.$$
It's determinant is $2$, which is coprime to $5 =$ `len(vowels)`.